In [4]:
%pip install pandas

%pip install joblib
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 1.5 MB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.4/39.4 MB 782.7 kB/s  0:00:45m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_csv("data/messages.csv")
print(df.shape)
print(df["split"].value_counts())

(360, 5)
split
train            200
test             100
stress_pidgin     25
stress_oos        20
stress_multi      15
Name: count, dtype: int64


In [3]:
core = df[df["split"].isin(["train", "test"])]
print(pd.crosstab(core["intent"], core["split"]))

split                 test  train
intent                           
complaint_refund        20     40
order_status            20     40
payment_confirmation    20     40
place_order             20     40
price_availability      20     40


#### Approach 1: Building the TF-IDF Classifier

In [5]:
import time

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline

df = pd.read_csv("data/messages.csv")
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

In [6]:
model = make_pipeline(
    TfidfVectorizer(lowercase=True, ngram_range=(1, 2), sublinear_tf=True),
    LogisticRegression(max_iter=1000),
)

In [7]:
start = time.perf_counter()
model.fit(train["message"], train["intent"])
print(f"Training time: {time.perf_counter() - start:.3f} seconds")

Training time: 0.123 seconds


#### Test

In [8]:
preds = model.predict(test["message"])
print(f"Test accuracy: {accuracy_score(test['intent'], preds):.3f}")
print(classification_report(test["intent"], preds))

Test accuracy: 0.890
                      precision    recall  f1-score   support

    complaint_refund       1.00      0.85      0.92        20
        order_status       0.78      0.90      0.84        20
payment_confirmation       0.90      0.90      0.90        20
         place_order       0.95      0.90      0.92        20
  price_availability       0.86      0.90      0.88        20

            accuracy                           0.89       100
           macro avg       0.90      0.89      0.89       100
        weighted avg       0.90      0.89      0.89       100



In [9]:
import numpy as np

vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]
names = np.array(vec.get_feature_names_out())
for intent, coefs in zip(clf.classes_, clf.coef_):
    top = names[np.argsort(coefs)[-5:][::-1]]
    print(f"{intent}: {', '.join(top)}")

complaint_refund: was, refund, food, the food, refund for
order_status: my order, my, the rider, is, order
payment_confirmation: confirm, please confirm, payment, paid, ve
place_order: to, to order, rice, of, order
price_availability: much, how much, do, do you, how


Some of these make sense, such as "refund", "paid", "how much" and "the rider". Others, like "was", "of" and "to", are filler words that happened to show up often in one intent in our small training set. That is a sign the model is partly learning quirks of 200 messages, and more labelled data would help.

#### How fast

In [10]:
start = time.perf_counter()
model.predict(["I've paid 12k, please confirm"])
print(f"Single prediction: {(time.perf_counter() - start) * 1000:.2f} ms")

Single prediction: 3.30 ms


In [ ]:
# Save for later
joblib.dump(model, "tfidf_model.joblib")